In [ ]:
import numpy as np


def mul(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    n = A.shape[0]
    C = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            for k in range(n):
                C[i, j] += A[i, k] * B[k, j]
    return C


def mul_rec(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    n = A.shape[0]
    if n <= 1:
        return A @ B

    mid = n // 2

    # 四个子矩阵
    A11, A12 = A[:mid, :mid], A[:mid, mid:]
    A21, A22 = A[mid:, :mid], A[mid:, mid:]
    B11, B12 = B[:mid, :mid], B[:mid, mid:]
    B21, B22 = B[mid:, :mid], B[mid:, mid:]

    C = np.empty_like(A)
    C[:mid, :mid] = mul_rec(A11, B11) + mul_rec(A12, B21)
    C[:mid, mid:] = mul_rec(A11, B12) + mul_rec(A12, B22)
    C[mid:, :mid] = mul_rec(A21, B11) + mul_rec(A22, B21)
    C[mid:, mid:] = mul_rec(A21, B12) + mul_rec(A22, B22)

    return C


def mul_rec_strassen(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    n = A.shape[0]
    if n <= 1:
        return A @ B

    mid = n // 2

    # 四个子矩阵
    A11, A12 = A[:mid, :mid], A[:mid, mid:]
    A21, A22 = A[mid:, :mid], A[mid:, mid:]
    B11, B12 = B[:mid, :mid], B[:mid, mid:]
    B21, B22 = B[mid:, :mid], B[mid:, mid:]

    # 7个子问题 递归调用
    T1 = mul_rec_strassen(A11, B11 + B21)
    T2 = mul_rec_strassen(A22, B12 + B22)
    T3 = mul_rec_strassen(A11 - A22, B12 + B21)
    T4 = mul_rec_strassen(A12 - A22, B22 - B21)
    T5 = mul_rec_strassen(A11 - A12, -B21)
    T6 = mul_rec_strassen(A11 - A21, B12 - B11)
    T7 = mul_rec_strassen(A21 - A22, B12)

    # 8次加减计算C
    C = np.empty((n, n))
    C[:mid, :mid] = T1 + T5
    C[:mid, mid:] = T2 + T3 + T4 + T5
    C[mid:, :mid] = T1 - T3 + T6 + T7
    C[mid:, mid:] = T2 + T7

    return C


m = n = 6

A, B = np.random.rand(n, n), np.random.rand(n, n)
C1 = mul(A, B)

# 如果n不是2的幂次 将m置为最小的大于n的2的幂次
if (n & (n - 1)) != 0:
    m |= m >> 1
    m |= m >> 2
    m |= m >> 4
    m |= m >> 8
    m |= m >> 16
    m += 1

# 补零
A = np.pad(A, ((0, m - n), (0, m - n)), 'constant')
B = np.pad(B, ((0, m - n), (0, m - n)), 'constant')

C2 = mul_rec(A, B)
C2 = C2[:n, :n]

C3 = mul_rec_strassen(A, B)
C3 = C3[:n, :n]

print("C1 = \n", C1)
print("C2 = \n", C2)
print("C3 = \n", C3)
print(np.allclose(C1, C2))
print(np.allclose(C1, C3))

C1 = 
 [[0.98475281 1.24629132 1.61182268 1.36782662 0.90435451 1.25882329]
 [2.13108141 2.34909866 2.30474127 2.82415514 1.24837663 1.97683245]
 [0.78839487 0.82026415 0.98806024 0.5477006  0.73167909 0.9193289 ]
 [1.77954907 2.02338249 1.85138118 1.82664733 1.00545504 1.54429732]
 [1.02242137 1.52605954 1.72378231 1.91522858 0.7519246  1.0488401 ]
 [1.19029061 2.03669178 2.0861378  1.9620111  0.68109698 1.08168987]]
C2 = 
 [[0.98475281 1.24629132 1.61182268 1.36782662 0.90435451 1.25882329]
 [2.13108141 2.34909866 2.30474127 2.82415514 1.24837663 1.97683245]
 [0.78839487 0.82026415 0.98806024 0.5477006  0.73167909 0.9193289 ]
 [1.77954907 2.02338249 1.85138118 1.82664733 1.00545504 1.54429732]
 [1.02242137 1.52605954 1.72378231 1.91522858 0.7519246  1.0488401 ]
 [1.19029061 2.03669178 2.0861378  1.9620111  0.68109698 1.08168987]]
C3 = 
 [[0.98475281 1.24629132 1.61182268 1.36782662 0.90435451 1.25882329]
 [2.13108141 2.34909866 2.30474127 2.82415514 1.24837663 1.97683245]
 [0.7883948